# 30 - Controlled Cohomology & Novikov Rings

When $\pi_1(M) \neq 0$, surgery theory changes character. The fundamental group $\pi$ acts on the universal cover $\tilde{M}$, and the correct homological invariants are those of $\tilde{M}$ as a $\mathbb{Z}[\pi]$-module, not just as an abelian group. This is **controlled** or **equivariant** cohomology.

The Novikov ring $\mathbb{Z}[\pi][t, t^{-1}]$ (or more generally the group ring completion) governs the surgery obstruction for non-simply-connected manifolds. The Wall $L$-groups $L_n(\mathbb{Z}[\pi])$ are Witt groups of such forms.

## Learning Goals
- Understand $\mathbb{Z}[\pi]$-modules and the universal cover as a module
- Build `FiniteGroupRing` and `UniversalCover` objects
- Compute $\mathbb{Z}[\pi]$-module cohomology with `compute_controlled_cohomology`
- Compute twisted intersection forms over $\mathbb{Z}[\pi]$ with `compute_twisted_intersection_form`
- Read `TwistedObstructionResult` and understand what vanishing means

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Non-simply-connected manifold $M$, $\pi_1(M) = \pi$ | `SimplicialComplex`, `FundamentalGroup` |
| **Algebraic** | $\mathbb{Z}[\pi]$-module $H_*(\tilde{M})$ | `TwistedChainComplex`, `FiniteGroupRing` |
| **Result** | Twisted obstruction in $L_n(\mathbb{Z}[\pi])$ | `TwistedObstructionResult` |

## Formal Background

For a group $\pi$ and a $\mathbb{Z}[\pi]$-module $M$, the **equivariant cohomology** is
$$H^k(\pi; M) = \mathrm{Ext}^k_{\mathbb{Z}[\pi]}(\mathbb{Z}, M).$$
For a manifold $M$ with $\pi_1 = \pi$, the cellular chain complex $C_*(\tilde{M})$ is a finitely generated free $\mathbb{Z}[\pi]$-module, and the surgery obstruction lives in $L_n(\mathbb{Z}[\pi])$, computed from the $\mathbb{Z}[\pi]$-valued intersection form on $H_{n/2}(\tilde{M})$.

In [ ]:
from pysurgery.core.controlled_cohomology import (
    FiniteGroupRing,
    UniversalCover,
    TwistedChainComplex,
    compute_controlled_cohomology, ControlledCohomologyResult,
    compute_twisted_intersection_form, TwistedIntersectionFormResult,
    compute_twisted_obstruction, TwistedObstructionResult,
)
from pysurgery.core.fundamental_group import FundamentalGroup
from pysurgery.core.complexes import SimplicialComplex

print("=" * 70)
print("30 - Controlled Cohomology & Novikov Rings: Setup Complete")
print("=" * 70)

## Part 1: The Group Ring $\mathbb{Z}[\pi]$

The **group ring** $\mathbb{Z}[\pi]$ is the free $\mathbb{Z}$-module with basis $\pi$, with multiplication induced by the group law. For $\pi = \mathbb{Z}/n\mathbb{Z}$ (cyclic group), $\mathbb{Z}[\mathbb{Z}/n] \cong \mathbb{Z}[t]/(t^n - 1)$.

`FiniteGroupRing` implements this algebraic structure and tracks deck transformations of the universal cover.

In [ ]:
# Cyclic group Z/3
pi1_C3 = FundamentalGroup.from_descriptor("C3")
GR_C3 = FiniteGroupRing(pi1_C3)

print("Group ring Z[Z/3]:")
print(f"  group:          {GR_C3.group_descriptor}")
print(f"  rank (as Z-mod): {GR_C3.rank}")
print(f"  generators:     {GR_C3.generators}")
print()

# The generator t satisfies t³ = 1
print(f"  Relation: t^{pi1_C3.order} = 1")
print()

# Quaternion group Q₈
try:
    pi1_Q8 = FundamentalGroup.from_descriptor("Q8")
    GR_Q8 = FiniteGroupRing(pi1_Q8)
    print(f"Group ring Z[Q₈]: rank={GR_Q8.rank}")
except Exception as e:
    print(f"Q₈ not available as built-in: {e}")
    # Use Z/4 as a simpler non-trivial example
    pi1_C4 = FundamentalGroup.from_descriptor("C4")
    GR_C4 = FiniteGroupRing(pi1_C4)
    print(f"Group ring Z[Z/4]: rank={GR_C4.rank}")

## Part 2: Universal Cover as a $\mathbb{Z}[\pi]$-Module

The **universal cover** $\tilde{M} \to M$ has a deck transformation action of $\pi_1(M) = \pi$. The cellular chain complex $C_*(\tilde{M})$ is therefore a free $\mathbb{Z}[\pi]$-module, with basis given by lifts of the cells of $M$.

`UniversalCover` constructs this module from a simplicial complex and its fundamental group.

In [ ]:
# Lens space L(3,1): π₁ = Z/3, a 3-dimensional non-simply-connected manifold
L31 = SimplicialComplex.lens_space(3, 1)

cover = UniversalCover(L31, pi1_C3)

print("Universal cover of L(3,1):")
print(f"  covering_degree: {cover.covering_degree}")
print(f"  cells_lifted:    {cover.cells_lifted}")
print(f"  π₁ generators:   {cover.deck_generators}")

# The Z[Z/3]-module chain complex
twisted_cx = TwistedChainComplex(cover, GR_C3)
print()
print("Twisted chain complex over Z[Z/3]:")
for k in range(4):
    try:
        rank = twisted_cx.free_rank(k)
        print(f"  C_{k}: free Z[Z/3]-rank = {rank}")
    except Exception:
        break

## Part 3: Controlled Cohomology

`compute_controlled_cohomology` computes $H^k(M; \mathbb{Z}[\pi])$ — the cohomology with coefficients in the group ring, which is the appropriate home for surgery obstructions in the non-simply-connected case.

The result is a `ControlledCohomologyResult` with the cohomology module as a $\mathbb{Z}[\pi]$-module (free rank + torsion over the group ring).

In [ ]:
for k in range(4):
    try:
        ctrl: ControlledCohomologyResult = compute_controlled_cohomology(
            twisted_cx, degree=k
        )
        print(f"H^{k}(L(3,1); Z[Z/3]):")
        print(f"  module_description: {ctrl.cohomology_module}")
        print(f"  free_rank:          {ctrl.free_rank}")
        print(f"  torsion_orders:     {ctrl.torsion_orders}")
        print(f"  exact:              {ctrl.exact}")
        print(f"  theorem_tag:        {ctrl.theorem_tag}")
        print()
    except Exception as e:
        print(f"H^{k}: {e}")
        break

## Part 4: Twisted Intersection Form

For an odd-dimensional manifold $M^{2k+1}$ with $\pi_1 = \pi$, the intersection form on the middle-dimensional homology is now a sesquilinear form over $\mathbb{Z}[\pi]$ with the anti-involution $\bar{g} = g^{-1}$.

`compute_twisted_intersection_form` returns a `TwistedIntersectionFormResult` with the form matrix over the group ring.

In [ ]:
tif: TwistedIntersectionFormResult = compute_twisted_intersection_form(
    L31, pi1_C3, GR_C3
)

print("Twisted intersection form for L(3,1):")
print(f"  matrix over Z[Z/3]:  {tif.matrix}")
print(f"  signature:           {tif.signature}")
print(f"  rank:                {tif.rank}")
print(f"  is_unimodular:       {tif.is_unimodular}")
print(f"  exact:               {tif.exact}")
print(f"  theorem_tag:         {tif.theorem_tag}")

## Part 5: Twisted Surgery Obstruction

`compute_twisted_obstruction` takes the twisted intersection form and computes the Wall obstruction element in $L_n(\mathbb{Z}[\pi])$. If it vanishes, the surgery problem is solvable and the manifold bounds a cobordism.

In [ ]:
obs: TwistedObstructionResult = compute_twisted_obstruction(tif, dimension=3)

print("Twisted surgery obstruction for L(3,1):")
print(f"  obstruction_class: {obs.obstruction_class}")
print(f"  vanishes:          {obs.vanishes}")
print(f"  explanation:       {obs.explanation}")
print(f"  exact:             {obs.exact}")
print(f"  theorem_tag:       {obs.theorem_tag}")
print()

# Comparison: S³ × S¹ (product manifold, trivially bounds)
try:
    S3xS1 = SimplicialComplex.product(SimplicialComplex.sphere(3), SimplicialComplex.circle())
    pi_free = FundamentalGroup.from_descriptor("Z")
    GR_Z = FiniteGroupRing(pi_free)
    cover_p = UniversalCover(S3xS1, pi_free)
    cx_p = TwistedChainComplex(cover_p, GR_Z)
    tif_p = compute_twisted_intersection_form(S3xS1, pi_free, GR_Z)
    obs_p = compute_twisted_obstruction(tif_p, dimension=4)
    print(f"S³×S¹ obstruction vanishes: {obs_p.vanishes}")
except Exception as e:
    print(f"(Product space computation: {e})")

## Summary Checklist

- [x] Understood $\mathbb{Z}[\pi]$-modules as the correct coefficients for non-simply-connected surgery
- [x] Built `FiniteGroupRing` and `UniversalCover` for cyclic groups
- [x] Computed controlled cohomology $H^k(M; \mathbb{Z}[\pi])$ with `TwistedChainComplex`
- [x] Computed twisted intersection forms with `compute_twisted_intersection_form`
- [x] Read the twisted surgery obstruction from `TwistedObstructionResult.vanishes`

## Next Steps
- **NB 09**: Fundamental group basics — the prerequisite for this notebook
- **NB 11**: Wall groups — the algebraic home of the obstructions computed here
- **NB 29**: Rational surgery — the CRT strategy for evaluating these obstructions
- **NB 34**: The capstone uses controlled cohomology for non-simply-connected inputs